In [7]:
STOCK_IDS: list[str] = [
    "1303", "4958", "6505", "8046", "3189", "4938", "1301", "3533", "1802", "6415",
    "1326", "2603", "2368", "2891", "3034", "3037", "2409", "3044", "3653", "2379",
    "2383", "2881", "3231", "2609", "2313", "6446", "2882", "2357", "3443", "3711",
    "2317", "3008", "2345", "2404", "2330", "2887", "3661", "2376", "2382", "2883",
]

# 設定全域股票範圍（whitelist 限制在這 40 檔）
# 之後所有 data.get() 與 data.indicator() 只會下載並回傳這 40 檔的資料
data.set_universe(whitelist=STOCK_IDS)

In [8]:
import contextlib, io, os, json, zipfile
from getpass import getpass
from pathlib import Path
import numpy as np
import pandas as pd
import finlab
from finlab import data
from IPython.display import display

finlab.login()  # 憑證已快取時不會再開瀏覽器

已登入（使用快取憑證）。


In [9]:
START_DATE = "2026-09-01"

1. 最新季 ROE (基本面)

In [10]:
def roe_latest():
    # 取得全體股票季ROE資料（不篩選特定個股、不排序）
    roe = data.get("fundamental_features:ROE稅後")

    # 取得指定季度的資料列，保留股票代號欄位標題與該列數據，並保留四捨五入
    roe_q = roe.loc[["2026-Q2"]].round(2)

    return roe_q

roe_latest()

symbol   1301  1303  1326  1802  2313  2317   2330   2345  2357   2368  ...  \
date                                                                    ...   
2026-Q2  2.21  6.13  1.34  2.43   3.1  3.41  11.39  16.42  6.29  12.41  ...   

symbol   3533  3653  3661  3711  4938  4958  6415  6446  6505  8046  
date                                                                 
2026-Q2  4.53  10.3  3.16  5.37  2.51  1.93  3.94   7.1  4.76  4.71  

[1 rows x 40 columns]

2. 最新月營收年增率 (基本面)

In [11]:
def revenue_yoy_latest():

    # 取得個股月營收年增率
    revenue = data.get("monthly_revenue:去年同月增減(%)", start=START_DATE)

    # 最新月份年增率，單位：%
    revenue_yoy_latest_pct = revenue.iloc[[-1]].round(2)

    return revenue_yoy_latest_pct
revenue_yoy_latest()

symbol      1301   1303   1326  1802   2313   2317   2330   2345   2357  \
date                                                                      
2026-10-12  7.13  55.25  44.77  34.1  15.01  38.41  54.64  66.12  17.73   

symbol       2368  ...   3533   3653    3661   3711   4938   4958  6415  \
date               ...                                                    
2026-10-12  55.46  ...  16.12  79.96  495.99  45.52  86.87  36.12  50.7   

symbol       6446   6505   8046  
date                             
2026-10-12  99.97  65.59  57.81  

[1 rows x 40 columns]

3. 最新月營收月增率 (基本面)

In [12]:
def revenue_mom_latest(stock_id="2330"):

    # 取得個股月營收月增率
    revenue = data.get("monthly_revenue:上月比較增減(%)", start=START_DATE)

    # 最新月份月增率，單位：%
    revenue_mom_latest_pct = revenue.iloc[[-1]].round(2)

    return revenue_mom_latest_pct
revenue_mom_latest()

symbol      1301   1303  1326  1802   2313   2317  2330  2345  2357   2368  \
date                                                                         
2026-10-12  7.33  12.49  9.29  8.18  22.89  25.69 -0.57 -1.15   2.4 -10.24   

symbol      ...  3533  3653  3661  3711    4938   4958  6415  6446  6505  8046  
date        ...                                                                 
2026-10-12  ...  4.95 -5.32  13.7  7.15  143.19  27.54   3.9  0.69  6.13  4.35  

[1 rows x 40 columns]

4. 收盤價 (技術面)

In [13]:
def close_latest():

    # 取得個股未還原收盤價
    close = data.get("price:收盤價", start=START_DATE)

    # 最新收盤價，單位：元／股
    close_twd = close.iloc[[-1]].round(2)

    return close_twd
close_latest()

symbol      1301   1303  1326  1802   2313   2317    2330    2345   2357  \
date                                                                       
2026-10-22  73.8  301.0  73.0  67.1  224.5  245.0  2385.0  1775.0  946.0   

symbol        2368  ...    3533    3653    3661   3711   4938   4958   6415  \
date                ...                                                       
2026-10-22  1115.0  ...  1795.0  6070.0  3495.0  654.0  103.5  567.0  472.0   

symbol        6446   6505    8046  
date                               
2026-10-22  1160.0  106.5  1265.0  

[1 rows x 40 columns]

5. 近5日報酬 (技術面)

In [14]:
def return_5d():

    # 取得個股還原收盤價
    p = data.get("etl:adj_close", start=START_DATE)

    # 近5日報酬，單位：%
    return_5d_pct = ((p.iloc[[-1]] / p.iloc[[-6]] - 1) * 100).round(2)

    return return_5d_pct
return_5d()

symbol      1301  1303  1326  1802  2313  2317  2330  2345  2357  2368  ...  \
date                                                                    ...   
2026-10-22 -9.67 -4.29 -5.19  7.02 -1.54   2.3 -2.25 -8.97 -2.97  -6.3  ...   

symbol      3533   3653  3661  3711  4938  4958  6415  6446  6505  8046  
date                                                                     
2026-10-22  -1.1 -10.27 -5.67 -2.82   5.5  9.25  1.18 -3.73 -4.48 -5.95  

[1 rows x 40 columns]

6. 近20日報酬 (技術面)

In [15]:
def return_20d():

    # 取得個股還原收盤價
    p = data.get("etl:adj_close", start=START_DATE)

    # 近20日報酬，單位：%
    return_20d_pct = ((p.iloc[[-1]] / p.iloc[[-21]] - 1) * 100).round(2)

    return return_20d_pct
return_20d()

symbol       1301   1303  1326   1802  2313  2317  2330  2345  2357  2368  \
date                                                                        
2026-10-22  13.19  25.68  7.67  11.65  1.13  -2.0 -3.83 -3.53   0.0   6.7   

symbol      ...   3533  3653  3661  3711   4938   4958   6415  6446   6505  \
date        ...                                                              
2026-10-22  ...  12.89  4.84 -4.64 -1.36  13.99  20.64  10.41  0.87  20.34   

symbol       8046  
date               
2026-10-22  19.34  

[1 rows x 40 columns]

7. 個股跟大盤近20相對報酬 (技術面)

In [16]:
import pandas as pd


def excess_return_20d():
    # 取得個股還原收盤價與大盤指數
    p = data.get("etl:adj_close", start=START_DATE)
    b = data.get("benchmark_return:發行量加權股價報酬指數", start=START_DATE)

    # 以收盤價 p 的 index 為基準對齊大盤，收盤價沒有的日期直接忽略
    b = b.squeeze("columns").reindex(p.index).ffill()

    # 計算最後一筆相較於前 20 個交易日的報酬率 (%)
    # 加上 .values 避免日期 index 不同而對齊出 NaN
    stock_return_20d = (p.iloc[[-1]] / p.iloc[-21].values - 1) * 100
    bench_return_20d = (b.iloc[-1] / b.iloc[-21] - 1) * 100

    # 個股報酬 - 大盤報酬
    excess_return_20d_pp = (stock_return_20d - bench_return_20d).round(2)

    return excess_return_20d_pp

excess_return_20d()

symbol      1301   1303   1326   1802  2313  2317  2330  2345  2357   2368  \
date                                                                         
2026-10-22  16.7  29.19  11.18  15.15  4.63  1.51 -0.32 -0.03  3.51  10.21   

symbol      ...  3533  3653  3661  3711   4938   4958   6415  6446   6505  \
date        ...                                                             
2026-10-22  ...  16.4  8.34 -1.13  2.15  17.49  24.15  13.92  4.38  23.85   

symbol       8046  
date               
2026-10-22  22.85  

[1 rows x 40 columns]

8. 5日均量／20日均量 (技術面)

In [17]:
def volume_ma5_to_ma20():

    # 取得個股成交股數
    volume = data.get("price:成交股數", start=START_DATE)

    # 計算5日與20日平均成交股數
    volume_ma5 = volume.rolling(5).mean().iloc[[-1]]
    volume_ma20 = volume.rolling(20).mean().iloc[[-1]]

    # 均量比，單位：倍
    volume_ma5_to_ma20_ratio = (volume_ma5 / volume_ma20).round(2)

    return volume_ma5_to_ma20_ratio
volume_ma5_to_ma20()

symbol      1301  1303  1326  1802  2313  2317  2330  2345  2357  2368  ...  \
date                                                                    ...   
2026-10-22   0.8  1.15  0.79  2.15  1.34  0.92   1.1  0.98  1.19  0.93  ...   

symbol      3533  3653  3661  3711  4938  4958  6415  6446  6505  8046  
date                                                                    
2026-10-22   0.6  1.08  0.57  0.96  1.09   0.9  1.77  1.07  1.17  0.75  

[1 rows x 40 columns]

9. 外資近5日買賣超強度 (籌碼面)

In [18]:
def foreign_net_buy_intensity_5d():

    # 取得個股成交股數
    volume = data.get("price:成交股數", start=START_DATE)

    # 取得外資買賣超股數
    foreign = data.get("institutional_investors_trading_summary:外陸資買賣超股數(不含外資自營商)", start=START_DATE)

    # 對齊交易日期
    foreign = foreign.reindex(volume.index)

    # 近5日外資買賣超股數／近5日成交股數，單位：%
    foreign_net_buy_intensity_5d_pct = (foreign.rolling(5).sum().iloc[[-1]] / volume.rolling(5).sum().iloc[[-1]] * 100).round(2)

    return foreign_net_buy_intensity_5d_pct
foreign_net_buy_intensity_5d()


symbol      1301  1303  1326  1802  2313   2317   2330  2345  2357  2368  ...  \
date                                                                      ...   
2026-10-22 -15.9 -5.31 -4.06  1.02 -2.52  22.88 -40.05 -5.93 -1.56 -9.49  ...   

symbol       3533  3653  3661   3711   4938  4958  6415  6446   6505   8046  
date                                                                         
2026-10-22 -12.15 -1.93 -7.41 -13.93  27.68 -3.27 -3.51 -6.02  14.86 -12.03  

[1 rows x 40 columns]

10. 外資最新交易日買賣超股數 (籌碼面)

In [19]:
def foreign_net_buy_1d(stock_id="2330"):

    # 取得外資買賣超股數
    foreign = data.get("institutional_investors_trading_summary:外陸資買賣超股數(不含外資自營商)", start=START_DATE)

    # 最新交易日買賣超，單位：股
    foreign_net_buy_1d_shares = foreign.iloc[[-1]]

    return foreign_net_buy_1d_shares
foreign_net_buy_1d()

symbol            1301        1303       1326        1802       2313  \
date                                                                   
2026-10-22 -16065504.0 -20416205.0 -8876465.0  25235718.0 -5006961.0   

symbol           2317        2330      2345      2357      2368  ...    3533  \
date                                                             ...           
2026-10-22  1739424.0 -12694626.0 -240640.0 -821452.0  368093.0  ... -8417.0   

symbol          3653      3661       3711       4938       4958      6415  \
date                                                                        
2026-10-22  311892.0 -107861.0 -5437744.0  1879147.0 -2981974.0 -389712.0   

symbol         6446        6505       8046  
date                                        
2026-10-22 -28400.0  12635429.0 -1033261.0  

[1 rows x 40 columns]

11. 投信近5日買賣超強度 (籌碼面)

In [20]:
def trust_net_buy_intensity_5d():

    # 取得個股成交股數
    volume = data.get("price:成交股數", start=START_DATE)

    # 取得投信買賣超股數
    trust = data.get("institutional_investors_trading_summary:投信買賣超股數", start=START_DATE)

    # 對齊交易日期
    trust = trust.reindex(volume.index)

    # 近5日投信買賣超股數／近5日成交股數，單位：%
    trust_net_buy_intensity_5d_pct = (trust.rolling(5).sum().iloc[[-1]] / volume.rolling(5).sum().iloc[[-1]] * 100).round(2)

    return trust_net_buy_intensity_5d_pct
trust_net_buy_intensity_5d()

symbol      1301  1303  1326  1802  2313  2317  2330  2345  2357   2368  ...  \
date                                                                     ...   
2026-10-22 -0.46  0.19  2.69  0.36  1.63  0.89  5.58  5.05 -5.87  10.21  ...   

symbol      3533  3653  3661  3711  4938  4958  6415  6446  6505  8046  
date                                                                    
2026-10-22  5.36  0.36  2.28  3.85 -4.82  2.26  0.29  2.71 -6.02  5.41  

[1 rows x 40 columns]

12. 投信最新交易日買賣超股數 (籌碼面)

In [21]:
def trust_net_buy_1d():

    # 取得投信買賣超股數
    trust = data.get("institutional_investors_trading_summary:投信買賣超股數", start=START_DATE)

    # 最新交易日買賣超，單位：股
    trust_net_buy_1d_shares = trust.iloc[[-1]]

    return trust_net_buy_1d_shares
trust_net_buy_1d()

symbol        1301      1303      1326       1802      2313      2317  \
date                                                                    
2026-10-22  1000.0 -475000.0  600000.0  2968000.0  141000.0 -255528.0   

symbol           2330     2345     2357      2368  ...  3533      3653  \
date                                               ...                   
2026-10-22  1349951.0  86000.0 -62400.0  497550.0  ...   0.0 -190500.0   

symbol        3661       3711      4938      4958     6415    6446       6505  \
date                                                                            
2026-10-22  1000.0  2251000.0 -740000.0  134000.0 -51000.0  1908.0 -8119000.0   

symbol          8046  
date                  
2026-10-22  289000.0  

[1 rows x 40 columns]

13. 20日平均成交金額 (技術面)

In [22]:
def amount_ma20():

    # 取得個股成交金額
    amount = data.get("price:成交金額", start=START_DATE)

    # 20日平均成交金額，保留原始金額單位
    amount_ma20_twd = (amount.rolling(20).mean().iloc[[-1]]).round(2)

    return amount_ma20_twd
amount_ma20()

symbol              1301          1303          1326          1802  \
date                                                                 
2026-10-22  6.150879e+09  2.829890e+10  3.166358e+09  5.986250e+09   

symbol              2313          2317          2330          2345  \
date                                                                 
2026-10-22  7.918830e+09  1.029193e+10  6.764174e+10  9.390124e+09   

symbol              2357          2368  ...          3533          3653  \
date                                    ...                               
2026-10-22  2.159238e+09  1.440030e+10  ...  3.303007e+09  8.679253e+09   

symbol              3661          3711          4938          4958  \
date                                                                 
2026-10-22  9.483299e+09  1.104161e+10  1.745864e+09  1.918376e+10   

symbol              6415          6446          6505          8046  
date                                                                
202

14. 10日報酬波動率 (技術面)

In [23]:
def volatility_10d():

    # 取得個股還原收盤價
    p = data.get("etl:adj_close", start=START_DATE)

    # 每日報酬
    daily_return = p / p.shift(1) - 1

    # 最近10日日報酬的樣本標準差，單位：%，未年化
    volatility_10d_daily_pct = (daily_return.rolling(10).std(ddof=1).iloc[[-1]] * 100).round(2)

    return volatility_10d_daily_pct
volatility_10d()

symbol      1301  1303  1326  1802  2313  2317  2330  2345  2357  2368  ...  \
date                                                                    ...   
2026-10-22  5.69  6.14  5.37  4.44  4.98  1.97  0.89  3.61  2.01  5.03  ...   

symbol      3533  3653  3661  3711  4938  4958  6415  6446  6505  8046  
date                                                                    
2026-10-22  3.11   3.4  3.81  3.94  3.78   4.7  4.91  2.57  6.33  3.18  

[1 rows x 40 columns]

15. 20日均線近5日變化率 (技術面)

In [24]:
def ma20_change_5d():

    # 取得個股還原收盤價
    p = data.get("etl:adj_close", start=START_DATE)

    # 計算20日均線
    ma20 = p.rolling(20).mean()

    # 最新20日均線相較5個交易日前的變化，單位：%
    ma20_change_5d_pct = ((ma20.iloc[[-1]] / ma20.iloc[[-6]] - 1) * 100).round(2)

    return ma20_change_5d_pct
ma20_change_5d()

symbol      1301  1303  1326  1802  2313  2317  2330  2345  2357  2368  ...  \
date                                                                    ...   
2026-10-22  4.76  6.88  2.88  3.54  1.98 -0.51 -0.15  0.96  0.79  4.48  ...   

symbol      3533  3653  3661  3711  4938  4958  6415  6446  6505  8046  
date                                                                    
2026-10-22  3.65   4.1  1.37  1.63  3.74  3.88  3.33 -0.11   7.0  4.71  

[1 rows x 40 columns]

16. 20日均線乖離率 (技術面)

In [25]:
def price_vs_ma20():

    # 取得個股還原收盤價
    p = data.get("etl:adj_close", start=START_DATE)

    # 最新20日均線
    ma20 = p.rolling(20).mean().iloc[[-1]]

    # 最新價格相對20日均線的乖離，單位：%
    price_vs_ma20_pct = ((p.iloc[-1] / ma20 - 1) * 100).round(2)

    return price_vs_ma20_pct
price_vs_ma20()

symbol      1301  1303  1326  1802  2313  2317  2330  2345  2357  2368  ...  \
date                                                                    ...   
2026-10-22   0.3  5.06 -0.53  7.45 -5.36  -1.3 -4.03 -9.15 -2.85 -5.41  ...   

symbol      3533   3653  3661  3711  4938  4958  6415  6446  6505  8046  
date                                                                     
2026-10-22 -1.17 -10.28 -9.51 -7.37  8.51  4.36  5.56 -1.92  2.18 -7.53  

[1 rows x 40 columns]

17. RSI14 (技術面)

In [27]:
def rsi_14():

    # 使用還原價格計算14日RSI
    rsi = data.indicator("RSI", adjust_price=True, timeperiod=14)

    # 最新RSI，範圍0～100
    rsi_14 = (rsi.iloc[[-1]]).round(2)

    return rsi_14
rsi_14()

             1301   1303   1326   1802   2313   2317   2330   2345   2357  \
date                                                                        
2026-10-22  51.61  56.57  50.32  60.55  44.08  47.35  37.91  37.42  45.09   

             2368  ...   3533   3653   3661   3711   4938   4958   6415  \
date               ...                                                    
2026-10-22  46.63  ...  50.82  42.88  39.31  43.98  67.49  56.74  55.83   

             6446   6505   8046  
date                             
2026-10-22  43.49  54.84  45.54  

[1 rows x 40 columns]